Author: Yicheng Rui (TDLI)

Email: ruiyicheng@sjtu.edu.cn

# ABC about camera
## Quantum efficiency
When the camera start to expose, the incoming photons would be converted to electrons by photoelectric effect. The ratio between generated photoelectron and incoming photon is called quantum efficiency $\eta_{QE}$. The higher quantum efficiency is, the faster the camera could collect photon.

\begin{equation}
\eta_{QE} = \frac{\text{Number of photoelectron of a pixel}}{\text{Number of incoming photons to this pixel}}
\end{equation}

## Gain
The camera would convert the electrons to numbers that computer can understand. These numbers are integers, whose unit is called analog-to-digital unit (ADU). The electron collected from an array of pixels would be converted into an array of number, which is a digital picture. The ratio between the collected number of photoelectron and the number of ADU of this pixel is called gain.

\begin{equation}
{\rm Gain} = \frac{\text{Number of photoelectron collected by a pixel}}{\text{Number of ADU of this pixel}}
\end{equation}
We need to notice that sometimes people would use the inverse of this expression to define Gain.

## Bias
For an unexposed pixel, the value for zero collected photoelectrons will translate, upon readout and analog-to-digital conversion, into a mean value with a small distribution about zero. To avoid negative numbers in the output image, electronics are set up to provide a positive offset value for eachaccumulated image. This offset value, the mean “zero” level, is called the bias level.


# What is image calibration?

Different pixels in a camera would have different quantum efficiency, different bias level and different size of effective aperture. These factors would affect the photometric precision. The basic idea of image calibration is to mitigate these effects using extra frames. 

## Bias field

Bias field is the image taken with zero exposure time. The configuration of the camera for the bias field should be the same as the scientific image, e.g. the sensor temperature, selected bias level.

## Flat field

Flat field is the image taken by exposing to a flat source. This source can be dome or the sky when the sun is rising or setting. The configuration of the camera for the flat field should also be the same as the scientific image. Empirically, the average ADU level of flat field should be half of the saturation level, which is 65535 for 16-bit camera.

## Signal-to-noise ratio 

Signal-to-noise ratio (SNR) is an important metric for assessing the quality of image. It is defined by the ratio betweeb a quantity and its uncertainty. For photometric system, the signal is the measured flux of a given star and the uncertainty of the flux mainly comes from scintillation, photon noise and fluctuation of the sky background. The SNR of photometry can be increased by stacking different frames. 

## Stacking

Stacking is to take the average (or median/maximum/minimum) of different image which taken for the same purpose. Stacking would increase the SNR of the image. Here is a simple explanation for why stacking would work. The central limit theorem is as follows: for a random variable $X$ with average value $\mu$ and standard deviation $\sigma$, its average value would satisfy

\begin{equation}
\lim_{N\to \infty} \frac{1}{N} \sum_i^{N} X_i \sim N (\mu,\frac{\sigma}{\sqrt{N}})
\end{equation}
where $N({\rm mean,std})$ is the normal destribution. When the number of image used for stacking $N$ becomes larger, the noise (uncertainty) of the average value would decrease as $\propto N^{-1/2}$. The more picture you use for stacking, the higher SNR you would obtain. This is the reason why we need to take a lot of dark frame and bias frame. We need to stack them to eliminate the fluctuations in bias field and photon noise in flat field.

## Method

The process of calibration is easy. The calibrated image can be obtained by

\begin{equation}
 \text{Calibration image} = \frac{\text{Raw science image - Bias field image}}{\text{Flat field image - Bias field image}}
\end{equation}

# Code for image calibration

## Step 1: Obtain the stacked Bias field and Flat field image

In [ ]:
#import packages
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob

In [ ]:
#To show a frame of the flat field and bias field
bias1data=fits.getdata('./input/bias/bias-0001.fit')
flat1data=fits.getdata('./input/flat/flat-0001.fit')
print('shape of image =',bias1data.shape)
f,axs=plt.subplots(1,2,figsize=(16,8))
axs[0].imshow(bias1data,vmin=395,vmax=405)
axs[0].set_title('bias')
axs[1].imshow(flat1data,vmin=20000,vmax=40000)
axs[1].set_title('flat')

In [ ]:
#Stacking bias
bias_paths = glob.glob('./input/bias/bias*.fit')
print('creating temp array')
bias_data = np.empty((len(bias_paths),*bias1data.shape),dtype = np.uint16)
print('reading image...')
for i,bias_path in enumerate(bias_paths):
    bias_data[i,:,:] = fits.getdata(bias_path)
print('taking the average...')
bias_stack = np.mean(bias_data,axis = 0)
print(bias_stack.shape)
fits.writeto('./output/bias.fit',bias_stack.astype('float32'),overwrite=True)
del bias_paths,bias_data #save memory

In [ ]:
f,axs=plt.subplots(1,2,figsize=(16,8))
axs[0].imshow(bias1data,vmin=395,vmax=405)
axs[0].set_title('single bias frame')
axs[1].imshow(bias_stack,vmin=395,vmax=405)
axs[1].set_title('stacked bias frame')

In [ ]:
# Stacking flat

flat_paths = glob.glob('./input/flat/flat*.fit')
print('creating temp array')
flat_data = np.empty((len(flat_paths),*bias1data.shape),dtype = np.uint16)
print('reading image...')
for i,flat_path in enumerate(flat_paths):
    flat_data[i,:,:] = fits.getdata(flat_path)-bias_stack
print('taking the average...')
flat_stack = np.mean(flat_data,axis = 0)
flat_stack = flat_stack/np.mean(flat_stack)
print(flat_stack.shape)
fits.writeto('./output/flat_no_bias.fit',flat_stack.astype('float32'),overwrite=True)
del flat_paths,flat_data #save memory

In [ ]:
f,axs=plt.subplots(1,2,figsize=(16,8))
axs[0].imshow((flat1data-bias_stack)/np.mean(flat1data),vmin=1,vmax=1.2)
axs[0].set_title('single flat frame')
axs[1].imshow(flat_stack,vmin=1,vmax=1.2)
axs[1].set_title('stacked flat frame')

Question: it seems that these two frames do not have a big difference. What's the possible reason for this? How to assess the improvement by stacking? What's the possible cause for the pattern?

## Step 2: Calibrate the scientific image using the stacked flat field and bias field

In [ ]:
science_paths = glob.glob('./input/science_image/wasp11*.fit')
print('processing image...')
for i,science_path in enumerate(science_paths):
    new_file_name = 'calibrate_'+science_path.split('/')[-1].split('.')[0]+'.fit'
    header_this = fits.getheader(science_path)
    data_this = (fits.getdata(science_path)-bias_stack)/flat_stack
    fits.writeto('./output/calibrated_image/'+new_file_name,data_this.astype('uint16'),header = header_this,overwrite=True)
    print('result saved at',new_file_name)

Now all the calibrated image are available in output/calibrated_image

In [ ]:
# view the result of calibration
f,axs=plt.subplots(1,2,figsize=(16,8))
axs[0].imshow(fits.getdata(science_path),vmin=1000,vmax=5000)
axs[0].set_title('frame before calibration')
axs[1].imshow(data_this,vmin=1000,vmax=5000)
axs[1].set_title('frame after calibration')

# You can use AstroImageJ to generate light curve for these images now!